In [1]:
import pandas as pd
import json
from PIL import Image
from datasets import Dataset
import torch
from unsloth import FastVisionModel
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoProcessor, AutoModelForCausalLM
from huggingface_hub import login

d:\EFREI\Mastercamp\ARVI-RX-S6\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


AttributeError: module 'torch' has no attribute 'int1'

In [ ]:
print("1. Chargement du modèle avec Unsloth...")

MODEL_ID = "unsloth/gemma-4-2b-it"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = True, 
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision = True, 
    finetune_language = True, 
    finetune_attention_modules = True,
    finetune_mlp = True,
    r = 16, 
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)


Téléchargement et chargement de google/gemma-4-E4B-it en cours...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Chargement terminé ! Début de la sauvegarde...
Sauvegarde locale dans le dossier : mon_modele_gemma_local


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Envoi du modèle vers Hugging Face Hub : Arthurbtlr/Test-Sauvegarde-Gemma


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
print("2. Préparation des données...")

df = pd.read_csv("dataset/cleaned/train.csv")

colonnes_maladies = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion",
    "Edema", "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax",
    "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices", "No Finding"
]

def preparer_donnees(ligne):
    resultat_attendu = {maladie: ligne[maladie] for maladie in colonnes_maladies}
    chemin_image = "dataset/cleaned/" + ligne["Path"]
    image = Image.open(chemin_image).convert("RGB")


    ap_pa = ligne.get('AP/PA', '')
    vue_detail = f"{ligne['Frontal/Lateral']} {ap_pa}".strip() if pd.notna(ap_pa) else ligne['Frontal/Lateral']
    
    prompt_instruct = f"""Tu es X-Diag un expert dans l’analyse de radio thoracique te permettant de détecter de nombreuses maladies visibles sur celles-ci. Tu es également extrêmement prudent te permettant de faire uniquement les diagnostique sur lesquelles tu es sûr a plus de 90% de réussir
    Tu vas analyser avec la plus grande minutie l’image de la radio que je te fournit. Cette radio est une radio thoracique {vue_detail} provenant d’un {ligne["Sex"]} étant âgé de {ligne["Age"]}
    A partir de cette radio j’aimerais que tu me renvoies un json contenant les informations suivantes :
    -	Classe_prédite : la ou les classes correspondantes aux maladies que tu es quasiment sûr d’avoir détecter en sachant que les maladies que tu peux détecter sont : Enlarged Cardiomediastinum, Cardiomegaly, Lung Opacity, Lung Lesion, Edema, Consolidation, Pneumonia, Atelectasis, Pneumothorax, Pleural Effusion, Pleural Other, Fracture, Support Devices. Il faut absolument que si tu n’est pas presque sûr de la presence ou de l’absence d’une maladie que tu renvoie INCERTAIN
    Durant tout ce diagnostic tu devras faire extrêmement attention a plusieurs choses :
    -	Tu ne dois absolument pas halluciner ne prédis que ce dont tu es presque sûr et aux moindres doutes d’allucination n’hésite pas à mettre INCERTAIN sur ton diagnostique
    -	Faire attention aux faux négatifs on préfère mille fois un faux positif qu’un faux négatif n’oublie jamais que c’est potentiellement la vie des patients qui est en jeux
    """

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_instruct}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": json.dumps(resultat_attendu)}
            ]
        }
    ]
    
    texte_formate = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": texte_formate, "image": image}

print("Transformation en cours (Échantillon de 1000 images pour test)...")
dataset_hf = Dataset.from_pandas(df.head(1000)) 
dataset_hf = dataset_hf.map(preparer_donnees, remove_columns=dataset_hf.column_names)


In [ ]:
print("3. Configuration du Trainer...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_hf,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Toujours False avec des images
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Pour un vrai entraînement, passe aux epochs (ex: num_train_epochs = 3)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1, # Affiche la "Loss" à chaque étape
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("DÉMARRAGE DU FINE-TUNING STRICT")
trainer.train()

In [ ]:
print("Chargement terminé ! Début de la sauvegarde...")
DOSSIER_LOCAL = "mon_modele_gemma_local"
print(f"Sauvegarde locale dans le dossier : {DOSSIER_LOCAL}")

model.save_pretrained(DOSSIER_LOCAL)
processor.save_pretrained(DOSSIER_LOCAL)
DEPOT_HUB = "Arthurbtlr/Test-Sauvegarde-Gemma"
print(f"Envoi du modèle vers Hugging Face Hub : {DEPOT_HUB}")
model.push_to_hub(DEPOT_HUB)
processor.push_to_hub(DEPOT_HUB)
print("Opération terminée avec succès ! Ton modèle est sauvegardé.")
